# Phase 2 — Churn Models (3 approaches compared)

We compare three churn-prediction approaches on the same time-based splits built in Phase 1:

| Approach | Library | Output |
|---|---|---|
| **GBM Stack** (XGB + LGBM + CatBoost + LR meta) | sklearn / xgboost / lightgbm / catboost | `p(churn)` |
| **BG-NBD** | lifetimes | `p(churn)` derived from expected purchases over 90d |
| **Cox PH Survival** | lifelines | `p(churn) = 1 - S(90d)` |

All three predict the same binary churn label so we compare with AUC-ROC, PR, F1.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.data.splits import build_windows
from src.features.build_features import build_customer_features
from src.models.churn.classification.stack import ChurnStack
from src.models.churn.bgnbd.model import BGNBDChurn, make_rft_summary
from src.models.churn.survival.model import CoxChurn, build_survival_frame
from src.evaluation.churn_metrics import evaluate, compare

PROCESSED = PROJECT_ROOT / "data" / "processed"
df = pd.read_parquet(PROCESSED / "transactions_clean.parquet")
labels = {
    name: pd.read_parquet(PROCESSED / f"churn_labels_{name}.parquet")
    for name in ["train", "val", "test"]
}
windows = build_windows(df, horizon_days=90, n_windows=3)
{w.name: (w.feature_end.date(), w.label_end.date()) for w in windows}

## Build features for each window

In [ ]:
top_countries = df["country"].value_counts().head(8).index.tolist()

features = {
    w.name: build_customer_features(df, w.feature_end, country_dummies=top_countries)
    for w in windows
}
for name, f in features.items():
    print(f"{name:>5}: {len(f):>6,} customers  |  {f.shape[1]} features")
features["train"].head(3)

In [ ]:
# Merge labels with features for each window
def join(name):
    return features[name].merge(labels[name][["customer_id", "churn"]], on="customer_id", how="inner")

data = {name: join(name) for name in ["train", "val", "test"]}
for name, d in data.items():
    print(f"{name:>5}: {len(d):>6,} rows  |  churn rate = {d['churn'].mean()*100:.2f}%")

## Model 1 — GBM Stack (Kaggle-style)

In [ ]:
feature_cols = [c for c in features["train"].columns if c not in {"customer_id", "country"}]
print(f"{len(feature_cols)} features used")

Xtr, ytr = data["train"][feature_cols].fillna(0), data["train"]["churn"]
Xva, yva = data["val"][feature_cols].fillna(0), data["val"]["churn"]
Xte, yte = data["test"][feature_cols].fillna(0), data["test"]["churn"]

stack = ChurnStack()
stack.fit(Xtr, ytr)
p_val_gbm = stack.predict_proba(Xva)
p_test_gbm = stack.predict_proba(Xte)

m_gbm_val = evaluate(yva.values, p_val_gbm)
m_gbm_test = evaluate(yte.values, p_test_gbm, threshold=m_gbm_val["threshold"])
m_gbm_val, m_gbm_test

## Model 2 — BG-NBD

In [ ]:
summary_tr = make_rft_summary(df, windows[0].feature_end)
summary_va = make_rft_summary(df, windows[1].feature_end)
summary_te = make_rft_summary(df, windows[2].feature_end)

bg = BGNBDChurn().fit(summary_tr)
p_val_bg = bg.churn_score(summary_va, horizon_days=90)
p_test_bg = bg.churn_score(summary_te, horizon_days=90)

# Align with the same customers used by GBM eval (those that have labels)
def aligned(score_series, label_df):
    s = score_series.reset_index().rename(columns={'index': 'customer_id'})
    s.columns = ['customer_id', 'score']
    merged = label_df[['customer_id', 'churn']].merge(s, on='customer_id', how='inner')
    return merged['churn'].values, merged['score'].values

y_va_bg, s_va_bg = aligned(p_val_bg, labels['val'])
y_te_bg, s_te_bg = aligned(p_test_bg, labels['test'])

m_bg_val = evaluate(y_va_bg, s_va_bg)
m_bg_test = evaluate(y_te_bg, s_te_bg, threshold=m_bg_val['threshold'])
m_bg_val, m_bg_test

## Model 3 — Cox PH Survival

In [ ]:
# Keep only numeric features for Cox (no categoricals — country was already one-hot)
numeric_feats = [c for c in feature_cols if pd.api.types.is_numeric_dtype(features['train'][c])]

surv_train = build_survival_frame(features['train'], labels['train'], numeric_feats)
cox = CoxChurn(penalizer=0.05).fit(surv_train)

# Predict on val and test feature tables (need same numeric columns)
p_val_cox = cox.churn_score(features['val'].set_index('customer_id')[numeric_feats], horizon_days=90)
p_test_cox = cox.churn_score(features['test'].set_index('customer_id')[numeric_feats], horizon_days=90)

y_va_cx, s_va_cx = aligned(p_val_cox, labels['val'])
y_te_cx, s_te_cx = aligned(p_test_cox, labels['test'])

m_cx_val = evaluate(y_va_cx, s_va_cx)
m_cx_test = evaluate(y_te_cx, s_te_cx, threshold=m_cx_val['threshold'])
m_cx_val, m_cx_test

## Comparison

In [ ]:
table = compare([
    {'name': 'GBM Stack — val', **m_gbm_val},
    {'name': 'GBM Stack — test', **m_gbm_test},
    {'name': 'BG-NBD — val', **m_bg_val},
    {'name': 'BG-NBD — test', **m_bg_test},
    {'name': 'Cox PH — val', **m_cx_val},
    {'name': 'Cox PH — test', **m_cx_test},
])
table

In [ ]:
# Persist scores for downstream use (decision layer, ablation report)
OUT = PROJECT_ROOT / 'data' / 'features'
OUT.mkdir(parents=True, exist_ok=True)

pd.DataFrame({'customer_id': data['test']['customer_id'], 'p_churn_gbm': p_test_gbm}).to_parquet(OUT / 'churn_scores_gbm_test.parquet', index=False)
pd.DataFrame({'customer_id': labels['test']['customer_id'], 'p_churn_bgnbd': p_test_bg.reindex(labels['test']['customer_id']).values}).to_parquet(OUT / 'churn_scores_bgnbd_test.parquet', index=False)
pd.DataFrame({'customer_id': labels['test']['customer_id'], 'p_churn_cox': p_test_cox.reindex(labels['test']['customer_id']).values}).to_parquet(OUT / 'churn_scores_cox_test.parquet', index=False)

table.to_csv(PROJECT_ROOT / 'reports' / 'churn_model_comparison.csv')
print('Saved scores and comparison table.')